# Gemini 4 Argon × MuJoCo による四足歩行ロボット (Petoi Bittle) の制御

このノートブックでは、[MarcHesse/bittle-mujoco](https://github.com/MarcHesse/bittle-mujoco) の物理シミュレーションモデルと **Gemini 4 Argon (`gemini-4-argon`)** を組み合わせて、自然言語から四足歩行ロボットを制御する3つのアプローチをステップバイステップで実装します。

### 全体構成
- **共通セットアップ**: ライブラリの読み込み、APIキーの確認、Notebook上での動画再生ヘルパーの定義
- **Part 1. プリセット動作を使った制御 (`bittle_gemini_control.py`)**
  - OpenCat の定義済み歩容（`trF`, `wkF` 等）や姿勢（`rest`, `sit` 等）を Python 関数としてラップし、Gemini の Tool Use（Function Calling）で順序立てて呼び出す方法
- **Part 2. プリセットを一切使わないゼロからの直接関節制御 (`bittle_gemini_direct.py`)**
  - プリセットを一切ロードせず、8個のサーボモーターの角度（肩・膝）を直感的な座標系に変換する「翻訳機」と、滑らかな補間・数式メトロノーム（CPG: 中枢パターン生成器）だけを用意し、Gemini 自身の物理・空間推論で「お座りして右前足でバイバイする」などの自由な動きをゼロから合成させる方法
- **Part 3. 自律スキル学習・蓄積・再利用システム (`bittle_gemini_skills.py`)**
  - Part 2 で Gemini が試行錯誤して見つけた「うまくいく関節軌道や歩行リズム」を `learned_skills.json` に名前付きスキルとして自律保存し、2回目以降は試行錯誤ゼロ・最小の API コールで即座に再利用・自己改良する仕組み


In [1]:
# 共通ライブラリのインポートと環境設定
import io
import json
import math
import os
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path
from typing import Any, Callable, Optional

import imageio.v2 as imageio
import imageio_ffmpeg
import mujoco
import numpy as np
from dotenv import load_dotenv
from google import genai
from google.genai import types
from IPython.display import HTML, Image as IPyImage, Video, display
from PIL import Image

ROOT_DIR = Path.cwd()
BITTLE_DIR = ROOT_DIR / "bittle-mujoco"
XML_PATH = BITTLE_DIR / "bittle.xml"

# .env から GEMINI_API_KEY をロード
load_dotenv(ROOT_DIR / ".env")
assert os.environ.get("GEMINI_API_KEY"), "GEMINI_API_KEY が .env に設定されていません。"
print("環境セットアップ完了！ XML_PATH:", XML_PATH)


環境セットアップ完了！ XML_PATH: /System/Volumes/Data/Data/haradatm/kenkyu/261024/bittle/bittle-mujoco/bittle.xml


---
## Part 1. プリセット動作を使った制御 (`bittle_gemini_control.py`)

最初のステップでは、`bittle-mujoco` リポジトリの `gaits.py` に収録されている **OpenCat のプリセット歩容（Gait）と姿勢（Pose）** を使います。

### 仕組みのポイント
1. **プリセットデータの読み込み (`gaits.py`)**
   - `convert_gait("trF")`：前進トロット歩行（48フレーム × 8関節角度）
   - `convert_pose("rest")`：伏せ姿勢（8関節角度）
2. **MuJoCo コントローラー (`BittleMuJoCoController`)**
   - 姿勢を変えるときは、急に角度を変えてロボットが跳ねないよう **コサイン補間 (`transition_to_pose`)** で滑らかに遷移させます。
   - 歩容を再生するときは、`0.02秒 (50Hz)` ごとにフレームを進め、最後に基本の立ち姿勢（`STAND_CTRL`）へスムーズに戻します。
3. **Gemini へのツール提供 (`build_gemini_tools`)**
   - `walk_forward(steps=3)` や `lie_down()` などの関数にわかりやすい英語の Docstring と型ヒントを付けるだけで、Gemini が自動的に適切な関数と引数を選んで順番に呼び出します。


In [2]:
# [Part 1 - Step 1] bittle-mujoco/gaits.py からプリセット変換関数を読み込む
if str(BITTLE_DIR) not in sys.path:
    sys.path.insert(0, str(BITTLE_DIR))

from gaits import GAITS, STAND_CTRL, _convert_frame, convert_gait, convert_pose, list_gaits, list_poses

print("利用可能なプリセット歩容 (Gaits):", list_gaits())
print("利用可能なプリセット姿勢 (Poses):", list_poses())
print("基本立ち姿勢 (STAND_CTRL) の8関節角度 [rad]:", np.round(STAND_CTRL, 4))


def convert_gait_right(name: str) -> np.ndarray:
    """左旋回プリセットの左右の脚カラムを入れ替えて右旋回歩容を作るヘルパー"""
    frames = GAITS[name]
    mirrored = [[f[1], f[0], f[3], f[2], f[5], f[4], f[7], f[6]] for f in frames]
    return np.array([_convert_frame(mf) for mf in mirrored])


利用可能なプリセット歩容 (Gaits): ['bdF', 'bkF', 'bkL', 'crF', 'crL', 'gpF', 'gpL', 'hlw', 'jpF', 'trF', 'trL', 'vtF', 'vtL', 'wkF', 'wkL']
利用可能なプリセット姿勢 (Poses): ['balance', 'buttUp', 'calib', 'dropped', 'lifted', 'lnd', 'rest', 'sit', 'str', 'up', 'zero']
基本立ち姿勢 (STAND_CTRL) の8関節角度 [rad]: [-0.7854  1.9199 -0.7854  1.9199  0.7854  1.9199  0.7854  1.9199]


In [3]:
# [Part 1 - Step 2] プリセットを再生する MuJoCo コントローラークラス
class BittleMuJoCoController:
    """OpenCatのプリセット歩容・姿勢を使って MuJoCo 上の Bittle を動かすコントローラー"""

    FRAME_DT = 0.020  # OpenCat のフレーム周期 (50 Hz)

    def __init__(self, xml_path: Path = XML_PATH, record_video: bool = True, fps: int = 30):
        self.model = mujoco.MjModel.from_xml_path(str(xml_path))
        self.data = mujoco.MjData(self.model)
        self.record_video = record_video
        self.fps = fps

        # カメラレンダラーの初期化 (640x480)
        # ※ MuJoCoのRenderer.close()は旧インスタンスGC時に自分のOpenGLコンテキストをmake_current()せずに
        #    mjr_context.free()を呼ぶバグがあり、新インスタンスのFBOを破壊してしまうため事前に安全にクローズする
        def _safe_renderer_close(r_self):
            if getattr(r_self, "_mjr_context", None) is not None:
                if getattr(r_self, "_gl_context", None) is not None:
                    r_self._gl_context.make_current()
                r_self._mjr_context.free()
                r_self._mjr_context = None
            if getattr(r_self, "_gl_context", None) is not None:
                r_self._gl_context.free()
                r_self._gl_context = None
        mujoco.Renderer.close = _safe_renderer_close
        prev_renderer = getattr(mujoco, "_active_bittle_renderer", None)
        if prev_renderer is not None:
            try:
                prev_renderer.close()
            except Exception:
                pass

        self.renderer = mujoco.Renderer(self.model, height=480, width=640)
        mujoco._active_bittle_renderer = self.renderer
        self.camera = mujoco.MjvCamera()
        mujoco.mjv_defaultFreeCamera(self.model, self.camera)
        self.camera.distance = 0.45
        self.camera.elevation = -22.0
        self.camera.azimuth = 135.0
        self.camera.lookat[:] = [0.0, 0.0, 0.05]

        self.frames: list[np.ndarray] = []
        self._last_render_time = -1.0
        self.current_ctrl = STAND_CTRL.copy()
        self.reset()

    def reset(self) -> dict:
        """XMLの stand キーフレームにリセットして 0.3秒間安定化させる"""
        mujoco.mj_resetDataKeyframe(self.model, self.data, 0)
        mujoco.mj_forward(self.model, self.data)
        self.current_ctrl = STAND_CTRL.copy()
        self.data.ctrl[:] = self.current_ctrl
        self._last_render_time = -1.0
        self._step_sim(0.3)
        return self.get_state()

    def _step_sim(self, duration_s: float, ctrl_fn: Optional[Callable[[float], np.ndarray]] = None):
        """指定秒数だけ物理シミュレーションを進める"""
        n_steps = int(round(duration_s / self.model.opt.timestep))
        for i in range(n_steps):
            t = i * self.model.opt.timestep
            if ctrl_fn is not None:
                self.current_ctrl = ctrl_fn(t)
            self.data.ctrl[:] = self.current_ctrl
            mujoco.mj_step(self.model, self.data)

            if self.record_video and (self.data.time - self._last_render_time >= 1.0 / self.fps):
                self._last_render_time = self.data.time
                self.frames.append(self.render_frame())

    def render_frame(self) -> np.ndarray:
        """ロボットの胴体を追従しながら現在のカメラ画像をレンダリングする"""
        torso_pos = self.data.sensor("torso_pos").data
        self.camera.lookat[0] = float(torso_pos[0])
        self.camera.lookat[1] = float(torso_pos[1])
        self.camera.lookat[2] = 0.05
        self.renderer.update_scene(self.data, camera=self.camera)
        return self.renderer.render().copy()

    def get_snapshot_png(self) -> bytes:
        """Gemini の視覚フィードバック用に PNG バイト列を取得する"""
        img = Image.fromarray(self.render_frame())
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        return buf.getvalue()

    def get_state(self) -> dict:
        """ロボットの3次元位置 (x, y, z) と姿勢角 (yaw, pitch, roll) を取得する"""
        pos = self.data.sensor("torso_pos").data.copy()
        w, x, y, z = self.data.sensor("torso_quat").data.copy()
        roll_deg = math.degrees(math.atan2(2.0 * (w * x + y * z), 1.0 - 2.0 * (x * x + y * y)))
        pitch_deg = math.degrees(math.asin(max(-1.0, min(1.0, 2.0 * (w * y - z * x)))))
        yaw_deg = math.degrees(math.atan2(2.0 * (w * z + x * y), 1.0 - 2.0 * (y * y + z * z)))
        return {
            "x_m": round(float(pos[0]), 4),
            "y_m": round(float(pos[1]), 4),
            "z_m": round(float(pos[2]), 4),
            "yaw_deg": round(yaw_deg, 1),
            "pitch_deg": round(pitch_deg, 1),
            "roll_deg": round(roll_deg, 1),
            "sim_time_s": round(float(self.data.time), 2),
        }

    def transition_to_pose(self, pose_name: str, transition_s: float = 0.6, hold_s: float = 0.6) -> dict:
        """現在の関節角度から目標のプリセット姿勢へコサイン補間で滑らかに遷移する"""
        start_state = self.get_state()
        target_ctrl = convert_pose(pose_name)
        start_ctrl = self.current_ctrl.copy()

        def blend_ctrl(t: float) -> np.ndarray:
            alpha = min(1.0, t / max(transition_s, 1e-4))
            smooth_alpha = 0.5 * (1.0 - math.cos(math.pi * alpha))
            return start_ctrl + (target_ctrl - start_ctrl) * smooth_alpha

        self._step_sim(transition_s + hold_s, ctrl_fn=blend_ctrl)
        return {"action": f"pose:{pose_name}", "before": start_state, "after": self.get_state()}

    def play_gait_cycles(self, gait_seq: np.ndarray, gait_label: str, steps: int = 1) -> dict:
        """プリセット歩容を指定サイクル（歩数）再生し、最後に立ち姿勢へ戻す"""
        steps = max(1, int(steps))
        start_state = self.get_state()
        if np.linalg.norm(self.current_ctrl - STAND_CTRL) > 0.15:
            self.transition_to_pose("balance", transition_s=0.4, hold_s=0.2)

        n_frames = len(gait_seq)
        total_duration = (n_frames * steps) * self.FRAME_DT
        warmup_time = min(0.4, total_duration * 0.3)

        def gait_ctrl(t: float) -> np.ndarray:
            frame_idx = int(t / self.FRAME_DT) % n_frames
            scale = min(1.0, t / max(warmup_time, 1e-4))
            return STAND_CTRL + (gait_seq[frame_idx] - STAND_CTRL) * scale

        self._step_sim(total_duration, ctrl_fn=gait_ctrl)

        # 歩行終了後、バランス姿勢に滑らかに戻す
        post_start = self.current_ctrl.copy()
        self._step_sim(0.35, ctrl_fn=lambda t: post_start + (STAND_CTRL - post_start) * min(1.0, t / 0.25))

        end_state = self.get_state()
        dist_m = math.hypot(end_state["x_m"] - start_state["x_m"], end_state["y_m"] - start_state["y_m"])
        return {
            "action": f"gait:{gait_label}",
            "steps": steps,
            "distance_moved_m": round(dist_m, 4),
            "before": start_state,
            "after": end_state,
        }

    def save_video(self, path: Path | str):
        """パイプ部分書き込みによるフレームずれ・色化けを防ぐため、一括バイト結合して ffmpeg に渡す"""
        if not self.frames:
            return
        import subprocess
        import imageio_ffmpeg
        path = Path(path)
        if path.suffix.lower() == ".gif":
            imageio.mimsave(str(path), self.frames, fps=self.fps)
            return
        h, w, _ = self.frames[0].shape
        raw_bytes = b"".join(np.ascontiguousarray(f, dtype=np.uint8).tobytes() for f in self.frames)
        cmd = [
            imageio_ffmpeg.get_ffmpeg_exe(),
            "-y",
            "-f", "rawvideo",
            "-vcodec", "rawvideo",
            "-s", f"{w}x{h}",
            "-pix_fmt", "rgb24",
            "-r", str(self.fps),
            "-i", "-",
            "-an",
            "-vcodec", "libx264",
            "-pix_fmt", "yuv420p",
            "-crf", "18",
            "-loglevel", "error",
            str(path),
        ]
        subprocess.run(cmd, input=raw_bytes, check=True)


In [4]:
# [Part 1 - Step 3] Gemini に公開するプリセット制御ツールとエージェントループの定義
def build_gemini_tools(controller: BittleMuJoCoController):
    def walk_forward(steps: int = 1, style: str = "trot") -> dict:
        """Move the Bittle robot forward by the specified number of steps (gait cycles)."""
        gait_map = {"trot": "trF", "walk": "wkF", "crawl": "crF", "gallop": "gpF"}
        return controller.play_gait_cycles(convert_gait(gait_map.get(style.lower(), "trF")), f"forward_{style}", steps=steps)

    def walk_backward(steps: int = 1) -> dict:
        """Move the Bittle robot backward by the specified number of steps."""
        return controller.play_gait_cycles(convert_gait("bkF"), "backward", steps=steps)

    def turn_left(steps: int = 1) -> dict:
        """Turn the Bittle robot to the left by the specified number of steps."""
        return controller.play_gait_cycles(convert_gait("trL"), "turn_left", steps=steps)

    def turn_right(steps: int = 1) -> dict:
        """Turn the Bittle robot to the right by the specified number of steps."""
        return controller.play_gait_cycles(convert_gait_right("trL"), "turn_right", steps=steps)

    def stand() -> dict:
        """Make the Bittle robot stand up in its balanced default posture."""
        return controller.transition_to_pose("balance", transition_s=0.6, hold_s=0.5)

    def lie_down() -> dict:
        """Make the Bittle robot lie down flat on the ground (rest / prone posture, '伏せ')."""
        return controller.transition_to_pose("rest", transition_s=0.8, hold_s=0.8)

    def sit() -> dict:
        """Make the Bittle robot sit down on its hind legs ('お座り')."""
        return controller.transition_to_pose("sit", transition_s=0.7, hold_s=0.7)

    tools = [walk_forward, walk_backward, turn_left, turn_right, stand, lie_down, sit]
    return tools, {fn.__name__: fn for fn in tools}


def run_preset_gemini_agent(prompt: str, controller: BittleMuJoCoController, model_name: str = "gemini-4-argon"):
    client = genai.Client()
    tools, tool_map = build_gemini_tools(controller)
    config = types.GenerateContentConfig(
        system_instruction=(
            "You are an embodied robotics controller driving a Petoi Bittle X quadruped in MuJoCo. "
            "Break down the user's natural language command into tool calls, verify the physical state "
            "and camera image after each step, and summarize the result in the user's language."
        ),
        tools=tools,
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
        temperature=0.1,
    )

    contents = [
        types.Content(
            role="user",
            parts=[
                types.Part.from_text(text=f"Initial state: {controller.get_state()}\nInstruction: {prompt}"),
                types.Part.from_bytes(data=controller.get_snapshot_png(), mime_type="image/png"),
            ],
        )
    ]

    for turn in range(1, 8):
        response = client.models.generate_content(model=model_name, contents=contents, config=config)
        candidate = response.candidates[0]
        contents.append(candidate.content)

        if not response.function_calls:
            print(f"\n[Final Answer (Turn {turn})]\n{response.text}")
            return response.text

        tool_parts = []
        for fc in response.function_calls:
            args = dict(fc.args) if fc.args else {}
            print(f"[Turn {turn}] Calling preset tool: {fc.name}({args})")
            res = tool_map[fc.name](**args)
            print(f"   -> Result: {res}")
            tool_parts.append(types.Part.from_function_response(name=fc.name, response={"result": res}))

        tool_parts.append(types.Part.from_bytes(data=controller.get_snapshot_png(), mime_type="image/png"))
        contents.append(types.Content(role="user", parts=tool_parts))


In [5]:
# [Part 1 - Step 4] 実行デモ：「前に3歩進んでから伏せて」
# ---------------------------------------------------------------------------
# 【.py スクリプトとしてターミナルから実行する場合のコマンド】
#   # MP4動画を保存して実行:
#   .venv/bin/python bittle_gemini_control.py --prompt "前に3歩進んでから伏せて" --video bittle_gemini_demo.mp4
#
#   # MuJoCoの3DインタラクティブViewerを起動しながら実行 (macOSは mjpython を使用):
#   .venv/bin/mjpython bittle_gemini_control.py --prompt "前に3歩進んでから伏せて" --viewer
# ---------------------------------------------------------------------------

preset_controller = BittleMuJoCoController(record_video=True)
run_preset_gemini_agent("前に3歩進んでから伏せて", preset_controller)

video_path_part1 = ROOT_DIR / "notebook_part1_preset.mp4"
preset_controller.save_video(video_path_part1)
Video(str(video_path_part1), embed=True, width=560)


[Turn 1] Calling preset tool: walk_forward({'steps': 3})
   -> Result: {'action': 'gait:forward_trot', 'steps': 3, 'distance_moved_m': 0.3222, 'before': {'x_m': 0.0006, 'y_m': -0.0, 'z_m': 0.0848, 'yaw_deg': -0.0, 'pitch_deg': -0.0, 'roll_deg': -0.0, 'sim_time_s': 0.3}, 'after': {'x_m': 0.3228, 'y_m': -0.0025, 'z_m': 0.0847, 'yaw_deg': -4.6, 'pitch_deg': 0.1, 'roll_deg': 0.1, 'sim_time_s': 3.53}}
[Turn 2] Calling preset tool: lie_down({})
   -> Result: {'action': 'pose:rest', 'before': {'x_m': 0.3228, 'y_m': -0.0025, 'z_m': 0.0847, 'yaw_deg': -4.6, 'pitch_deg': 0.1, 'roll_deg': 0.1, 'sim_time_s': 3.53}, 'after': {'x_m': 0.312, 'y_m': -0.0014, 'z_m': 0.0357, 'yaw_deg': -4.8, 'pitch_deg': -0.0, 'roll_deg': -0.0, 'sim_time_s': 5.13}}

[Final Answer (Turn 3)]
前に3歩進み（約0.32m前進）、その場で伏せ（休止姿勢）の動作を完了しました。

- **最終位置**: `(x, y, z) = (0.312m, -0.0014m, 0.0357m)`
- **姿勢**: 安定した伏せ状態（`pitch: 0.0°`, `roll: 0.0°`）


In [6]:
preset_controller = BittleMuJoCoController(record_video=True)
run_preset_gemini_agent("後ろ足のだけ2足立ちして、両前足を上げて3回バイバイして", preset_controller)

video_path_part1 = ROOT_DIR / "notebook_part1_preset_biped.mp4"
preset_controller.save_video(video_path_part1)
Video(str(video_path_part1), embed=True, width=560)


[Final Answer (Turn 1)]
申し訳ありませんが、現在のコントローラーには後ろ足での2足立ちや、両前足を上げて手を振る（バイバイする）ための動作ツールが用意されていません。

現在実行可能な基本動作は以下の通りです：
- **前進 / 後退** (`walk_forward` / `walk_backward`)
- **左旋回 / 右旋回** (`turn_left` / `turn_right`)
- **立つ** (`stand`)
- **お座り** (`sit`)
- **伏せ** (`lie_down`)

もしよろしければ、「お座り（後ろ足を曲げて上体を起こす姿勢）」などで代用するか、他の移動・姿勢コマンドをご指示ください。


---
## Part 2. プリセットを一切使わないゼロからの直接関節制御 (`bittle_gemini_direct.py`)

Part 1 ではあらかじめ人間が作った `gaits.py` の角度表を再生していましたが、これだと **プリセットに無い動作（例：「右前足を上げてバイバイ（手を振る）」「腕立て伏せ」など）** はできません。

そこで Part 2 では **`gaits.py` を一切ロードせず**、Gemini 4 Argon に 8個のサーボモーター角度をゼロから設計させます。

### 直感的に動かすための3つの工夫
1. **モーター回転方向の「翻訳機」 (`symmetric_deg_to_mjcf_ctrl`)**
   - `bittle.xml` の生データでは、前足と後ろ足でモーター回転軸のプラス・マイナスが逆になっています。
   - そこで、4本すべての脚（`RF`, `LF`, `RR`, `LR`）で **「ニュートラル立ち姿勢＝`0°`、太ももを前に振る＝`+shoulder_deg`、すねを前上に持ち上げる＝`+knee_deg`」** という統一ルールに変換します。
2. **キーフレーム補間器 (`execute_keyframe_trajectory`)**
   - アニメの「原画（キーフレーム）」のように、Gemini が数枚の関節角度リストを指定すると、その間をコサイン曲線で滑らかに繋いで再生します。これで「お座り → 片足を上げて3回振る → 伏せる」といった自在なジェスチャーが作れます。
3. **数式メトロノーム CPG (`execute_custom_oscillator`)**
   - 歩行のような周期運動のために、サイン波・コサイン波による中枢パターン生成器（CPG）を用意し、**「どの脚をどの位相（タイミング `phase_deg`）と振り幅（`amp_deg`）で動かすか」** の数式パラメータを Gemini 自身に設計させます。


In [7]:
# [Part 2 - Step 1] プリセットを使わない「8関節の対称角度（度数法）⇔ MuJoCoラジアン」翻訳関数
def symmetric_deg_to_mjcf_ctrl(
    rf_shoulder_deg: float,
    rf_knee_deg: float,
    lf_shoulder_deg: float,
    lf_knee_deg: float,
    rr_shoulder_deg: float,
    rr_knee_deg: float,
    lr_shoulder_deg: float,
    lr_knee_deg: float,
    stand_ctrl: np.ndarray,
) -> np.ndarray:
    """4脚共通の直感的な相対角度（+が前・上）を MuJoCo アクチュエータのラジアン指令値に変換する"""
    ctrl = stand_ctrl.copy()
    # 前足(0, 2)と後ろ足(4, 6)で逆になっている回転軸の符号を吸収し、「+deg = 太ももを前に振り出す」に統一
    ctrl[0] = stand_ctrl[0] - math.radians(rf_shoulder_deg)  # 右前・肩
    ctrl[2] = stand_ctrl[2] - math.radians(lf_shoulder_deg)  # 左前・肩
    ctrl[4] = stand_ctrl[4] + math.radians(rr_shoulder_deg)  # 右後・肩
    ctrl[6] = stand_ctrl[6] + math.radians(lr_shoulder_deg)  # 左後・肩

    # 膝(1, 3, 5, 7)はすべて「+deg = すねを前・上方向へ持ち上げる」に統一
    ctrl[1] = stand_ctrl[1] - math.radians(rf_knee_deg)      # 右前・膝
    ctrl[3] = stand_ctrl[3] - math.radians(lf_knee_deg)      # 左前・膝
    ctrl[5] = stand_ctrl[5] - math.radians(rr_knee_deg)      # 右後・膝
    ctrl[7] = stand_ctrl[7] - math.radians(lr_knee_deg)      # 左後・膝
    return np.clip(ctrl, -2.6, 2.6)


def mjcf_ctrl_to_symmetric_deg(ctrl: np.ndarray, stand_ctrl: np.ndarray) -> dict[str, float]:
    """現在の MuJoCo ラジアン指令値を直感的な度数法に変換して Gemini にフィードバックする"""
    return {
        "rf_shoulder_deg": round(math.degrees(stand_ctrl[0] - ctrl[0]), 1),
        "rf_knee_deg": round(math.degrees(stand_ctrl[1] - ctrl[1]), 1),
        "lf_shoulder_deg": round(math.degrees(stand_ctrl[2] - ctrl[2]), 1),
        "lf_knee_deg": round(math.degrees(stand_ctrl[3] - ctrl[3]), 1),
        "rr_shoulder_deg": round(math.degrees(ctrl[4] - stand_ctrl[4]), 1),
        "rr_knee_deg": round(math.degrees(stand_ctrl[5] - ctrl[5]), 1),
        "lr_shoulder_deg": round(math.degrees(ctrl[6] - stand_ctrl[6]), 1),
        "lr_knee_deg": round(math.degrees(stand_ctrl[7] - ctrl[7]), 1),
    }


In [8]:
# [Part 2 - Step 2] プリセットを一切持たない物理サンドボックス `DirectBittleSandbox`
class DirectBittleSandbox:
    """gaits.py を一切使わず、直接8関節の軌道とCPGリズムを物理実行するサンドボックス"""

    def __init__(self, xml_path: Path = XML_PATH, record_video: bool = True, fps: int = 30):
        self.model = mujoco.MjModel.from_xml_path(str(xml_path))
        self.data = mujoco.MjData(self.model)

        # bittle.xml のハードウェア初期キャリブレーション姿勢 (keyframe 0) だけを基準点として取得
        self.stand_ctrl = self.model.key_ctrl[0].copy()
        self.current_ctrl = self.stand_ctrl.copy()

        self.record_video = record_video
        self.fps = fps

        # ※ MuJoCoのRenderer.close()は旧インスタンスGC時に自分のOpenGLコンテキストをmake_current()せずに
        #    mjr_context.free()を呼ぶバグがあり、新インスタンスのFBOを破壊してしまうため事前に安全にクローズする
        def _safe_renderer_close(r_self):
            if getattr(r_self, "_mjr_context", None) is not None:
                if getattr(r_self, "_gl_context", None) is not None:
                    r_self._gl_context.make_current()
                r_self._mjr_context.free()
                r_self._mjr_context = None
            if getattr(r_self, "_gl_context", None) is not None:
                r_self._gl_context.free()
                r_self._gl_context = None
        mujoco.Renderer.close = _safe_renderer_close
        prev_renderer = getattr(mujoco, "_active_bittle_renderer", None)
        if prev_renderer is not None:
            try:
                prev_renderer.close()
            except Exception:
                pass

        self.renderer = mujoco.Renderer(self.model, height=480, width=640)
        mujoco._active_bittle_renderer = self.renderer
        self.camera = mujoco.MjvCamera()
        mujoco.mjv_defaultFreeCamera(self.model, self.camera)
        self.camera.distance = 0.45
        self.camera.elevation = -20.0
        self.camera.azimuth = 135.0
        self.camera.lookat[:] = [0.0, 0.0, 0.05]

        self.frames: list[np.ndarray] = []
        self._last_render_time = -1.0
        self.reset()

    def reset(self) -> dict:
        mujoco.mj_resetDataKeyframe(self.model, self.data, 0)
        mujoco.mj_forward(self.model, self.data)
        self.current_ctrl = self.stand_ctrl.copy()
        self.data.ctrl[:] = self.current_ctrl
        self._last_render_time = -1.0
        self._step_sim(0.3)
        return self.get_state()

    def _step_sim(self, duration_s: float, ctrl_fn: Optional[Callable[[float], np.ndarray]] = None):
        n_steps = max(1, int(round(duration_s / self.model.opt.timestep)))
        for i in range(n_steps):
            t = i * self.model.opt.timestep
            if ctrl_fn is not None:
                self.current_ctrl = ctrl_fn(t)
            self.data.ctrl[:] = self.current_ctrl
            mujoco.mj_step(self.model, self.data)

            if self.record_video and (self.data.time - self._last_render_time >= 1.0 / self.fps):
                self._last_render_time = self.data.time
                self.frames.append(self.render_frame())

    def render_frame(self) -> np.ndarray:
        torso_pos = self.data.sensor("torso_pos").data
        self.camera.lookat[0] = float(torso_pos[0])
        self.camera.lookat[1] = float(torso_pos[1])
        self.camera.lookat[2] = 0.05
        self.renderer.update_scene(self.data, camera=self.camera)
        return self.renderer.render().copy()

    def get_snapshot_png(self) -> bytes:
        img = Image.fromarray(self.render_frame())
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        return buf.getvalue()

    def get_state(self) -> dict:
        """胴体の位置・床からの高さ・姿勢角・転倒有無・各足先の高さを返す"""
        pos = self.data.sensor("torso_pos").data.copy()
        w, x, y, z = self.data.sensor("torso_quat").data.copy()
        roll_deg = math.degrees(math.atan2(2.0 * (w * x + y * z), 1.0 - 2.0 * (x * x + y * y)))
        pitch_deg = math.degrees(math.asin(max(-1.0, min(1.0, 2.0 * (w * y - z * x)))))
        yaw_deg = math.degrees(math.atan2(2.0 * (w * z + x * y), 1.0 - 2.0 * (y * y + z * z)))

        floor_z = float(self.data.geom_xpos[self.model.geom("floor").id][2])
        foot_heights_mm = {
            leg: round((float(self.data.site(site).xpos[2]) - floor_z) * 1000.0, 1)
            for leg, site in [("RF", "rf_foot_site"), ("LF", "lf_foot_site"), ("RR", "rr_foot_site"), ("LR", "lr_foot_site")]
        }
        return {
            "torso_pos_m": {
                "x": round(float(pos[0]), 4),
                "y": round(float(pos[1]), 4),
                "z_above_floor": round(float(pos[2]) - floor_z, 4),
            },
            "orientation_deg": {"yaw": round(yaw_deg, 1), "pitch": round(pitch_deg, 1), "roll": round(roll_deg, 1)},
            "fallen": abs(roll_deg) > 60.0 or abs(pitch_deg) > 60.0,
            "commanded_joint_angles_deg": mjcf_ctrl_to_symmetric_deg(self.current_ctrl, self.stand_ctrl),
            "foot_site_heights_above_floor_mm": foot_heights_mm,
            "sim_time_s": round(float(self.data.time), 2),
        }

    def move_joints_to(
        self,
        rf_shoulder_deg: float = 0.0, rf_knee_deg: float = 0.0,
        lf_shoulder_deg: float = 0.0, lf_knee_deg: float = 0.0,
        rr_shoulder_deg: float = 0.0, rr_knee_deg: float = 0.0,
        lr_shoulder_deg: float = 0.0, lr_knee_deg: float = 0.0,
        transition_s: float = 0.5, hold_s: float = 0.3,
    ) -> dict:
        """指定された8関節角度へコサイン補間で滑らかに移動する"""
        before = self.get_state()
        start_ctrl = self.current_ctrl.copy()
        target_ctrl = symmetric_deg_to_mjcf_ctrl(
            rf_shoulder_deg, rf_knee_deg, lf_shoulder_deg, lf_knee_deg,
            rr_shoulder_deg, rr_knee_deg, lr_shoulder_deg, lr_knee_deg,
            stand_ctrl=self.stand_ctrl,
        )
        transition_s = max(0.05, float(transition_s))
        hold_s = max(0.0, float(hold_s))

        def interp_ctrl(t: float) -> np.ndarray:
            alpha = min(1.0, t / transition_s)
            return start_ctrl + (target_ctrl - start_ctrl) * (0.5 * (1.0 - math.cos(math.pi * alpha)))

        self._step_sim(transition_s + hold_s, ctrl_fn=interp_ctrl)
        return {"before_pos_m": before["torso_pos_m"], "after_state": self.get_state()}

    def execute_keyframe_trajectory(self, keyframes: list[dict]) -> dict:
        """Geminiが設計した複数フレームの関節角度リストを順番に滑らかに実行する"""
        before = self.get_state()
        for kf in keyframes:
            self.move_joints_to(
                rf_shoulder_deg=float(kf.get("rf_shoulder_deg", 0.0)),
                rf_knee_deg=float(kf.get("rf_knee_deg", 0.0)),
                lf_shoulder_deg=float(kf.get("lf_shoulder_deg", 0.0)),
                lf_knee_deg=float(kf.get("lf_knee_deg", 0.0)),
                rr_shoulder_deg=float(kf.get("rr_shoulder_deg", 0.0)),
                rr_knee_deg=float(kf.get("rr_knee_deg", 0.0)),
                lr_shoulder_deg=float(kf.get("lr_shoulder_deg", 0.0)),
                lr_knee_deg=float(kf.get("lr_knee_deg", 0.0)),
                transition_s=float(kf.get("transition_s", 0.25)),
                hold_s=float(kf.get("hold_s", 0.05)),
            )
        after = self.get_state()
        return {
            "num_keyframes_executed": len(keyframes),
            "displacement_m": {
                "dx": round(after["torso_pos_m"]["x"] - before["torso_pos_m"]["x"], 4),
                "dy": round(after["torso_pos_m"]["y"] - before["torso_pos_m"]["y"], 4),
            },
            "before_pos_m": before["torso_pos_m"],
            "after_state": after,
        }

    def execute_custom_oscillator(
        self,
        cycles: int, cycle_period_s: float,
        rf_phase_deg: float, lf_phase_deg: float, rr_phase_deg: float, lr_phase_deg: float,
        rf_shoulder_amp_deg: float, lf_shoulder_amp_deg: float, rr_shoulder_amp_deg: float, lr_shoulder_amp_deg: float,
        knee_lift_amp_deg: float, shoulder_offset_deg: float = 0.0, knee_offset_deg: float = 0.0,
    ) -> dict:
        """サイン波・コサイン波による CPG（中枢パターン生成器）で周期的な歩容を合成・実行する"""
        before = self.get_state()
        # お座りや伏せの姿勢から歩き始める場合は、まず中立姿勢に0.45秒かけて整列させて転倒を防ぐ
        if np.linalg.norm(self.current_ctrl - self.stand_ctrl) > 0.15:
            pre_start = self.current_ctrl.copy()
            target_stand = symmetric_deg_to_mjcf_ctrl(
                shoulder_offset_deg, knee_offset_deg, shoulder_offset_deg, knee_offset_deg,
                shoulder_offset_deg, knee_offset_deg, shoulder_offset_deg, knee_offset_deg,
                stand_ctrl=self.stand_ctrl,
            )
            self._step_sim(
                0.45,
                ctrl_fn=lambda t: pre_start + (target_stand - pre_start) * (0.5 * (1.0 - math.cos(math.pi * min(1.0, t / 0.35)))),
            )

        cycles = max(1, int(cycles))
        cycle_period_s = max(0.2, float(cycle_period_s))
        total_time = cycles * cycle_period_s
        warmup_s = min(0.3, total_time * 0.25)

        phases = {k: math.radians(v) for k, v in [("rf", rf_phase_deg), ("lf", lf_phase_deg), ("rr", rr_phase_deg), ("lr", lr_phase_deg)]}
        sh_amps = {"rf": rf_shoulder_amp_deg, "lf": lf_shoulder_amp_deg, "rr": rr_shoulder_amp_deg, "lr": lr_shoulder_amp_deg}

        def cpg_ctrl(t: float) -> np.ndarray:
            env = min(1.0, t / max(warmup_s, 1e-4))
            omega = 2.0 * math.pi / cycle_period_s
            degs = {}
            for leg in ("rf", "lf", "rr", "lr"):
                theta = omega * t + phases[leg]
                degs[f"{leg}_sh"] = shoulder_offset_deg - sh_amps[leg] * math.cos(theta) * env
                degs[f"{leg}_kn"] = knee_offset_deg + knee_lift_amp_deg * max(0.0, math.sin(theta)) * env
            return symmetric_deg_to_mjcf_ctrl(
                degs["rf_sh"], degs["rf_kn"], degs["lf_sh"], degs["lf_kn"],
                degs["rr_sh"], degs["rr_kn"], degs["lr_sh"], degs["lr_kn"],
                stand_ctrl=self.stand_ctrl,
            )

        self._step_sim(total_time, ctrl_fn=cpg_ctrl)
        after = self.get_state()
        return {
            "cycles": cycles,
            "displacement_m": {
                "dx": round(after["torso_pos_m"]["x"] - before["torso_pos_m"]["x"], 4),
                "dy": round(after["torso_pos_m"]["y"] - before["torso_pos_m"]["y"], 4),
            },
            "before_pos_m": before["torso_pos_m"],
            "after_state": after,
        }

    def save_video(self, path: Path | str):
        """パイプ部分書き込みによるフレームずれ・色化けを防ぐため、一括バイト結合して ffmpeg に渡す"""
        if not self.frames:
            return
        import subprocess
        import imageio_ffmpeg
        path = Path(path)
        if path.suffix.lower() == ".gif":
            imageio.mimsave(str(path), self.frames, fps=self.fps)
            return
        h, w, _ = self.frames[0].shape
        raw_bytes = b"".join(np.ascontiguousarray(f, dtype=np.uint8).tobytes() for f in self.frames)
        cmd = [
            imageio_ffmpeg.get_ffmpeg_exe(),
            "-y",
            "-f", "rawvideo",
            "-vcodec", "rawvideo",
            "-s", f"{w}x{h}",
            "-pix_fmt", "rgb24",
            "-r", str(self.fps),
            "-i", "-",
            "-an",
            "-vcodec", "libx264",
            "-pix_fmt", "yuv420p",
            "-crf", "18",
            "-loglevel", "error",
            str(path),
        ]
        subprocess.run(cmd, input=raw_bytes, check=True)


In [9]:
# [Part 2 - Step 3] プリセットを使わない低レベル関節制御ツールとエージェントの定義
from bittle_gemini_direct import SYSTEM_INSTRUCTION as DIRECT_SYSTEM_INSTRUCTION, build_direct_tools as build_pure_direct_tools


def run_pure_direct_agent(prompt: str, sandbox: DirectBittleSandbox, model_name: str = "gemini-4-argon"):
    client = genai.Client()
    tools, tool_map = build_pure_direct_tools(sandbox)
    config = types.GenerateContentConfig(
        system_instruction=DIRECT_SYSTEM_INSTRUCTION,
        tools=tools,
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
        temperature=0.1,
    )

    contents = [
        types.Content(
            role="user",
            parts=[
                types.Part.from_text(text=f"Initial robot state: {sandbox.get_state()}\nUser instruction: {prompt}"),
                types.Part.from_bytes(data=sandbox.get_snapshot_png(), mime_type="image/png"),
            ],
        )
    ]

    for turn in range(1, 8):
        response = client.models.generate_content(model=model_name, contents=contents, config=config)
        candidate = response.candidates[0]
        contents.append(candidate.content)

        if not response.function_calls:
            print(f"\n[Final Answer (Turn {turn})]\n{response.text}")
            return response.text

        tool_parts = []
        for fc in response.function_calls:
            args = dict(fc.args) if fc.args else {}
            print(f"[Turn {turn}] Calling low-level tool: {fc.name}({args})")
            res = tool_map[fc.name](**args)
            print(f"   -> Result: displacement={res.get('displacement_m')}, after_pos={res.get('after_state', {}).get('torso_pos_m')}")
            tool_parts.append(types.Part.from_function_response(name=fc.name, response={"result": res}))

        tool_parts.append(types.Part.from_bytes(data=sandbox.get_snapshot_png(), mime_type="image/png"))
        contents.append(types.Content(role="user", parts=tool_parts))


In [10]:
# [Part 2 - Step 4] 実行デモ：プリセットに存在しない「お座りして右前足で3回バイバイ」をゼロから合成
# ---------------------------------------------------------------------------
# 【.py スクリプトとしてターミナルから実行する場合のコマンド】
#   # MP4動画を保存して実行:
#   .venv/bin/python bittle_gemini_direct.py \
#     --prompt "前に3歩進んでから、お座りして、右前足を上げて3回バイバイ（手を振る）して、最後に伏せて" \
#     --video bittle_direct_demo.mp4
#
#   # MuJoCoの3DインタラクティブViewerを起動しながら実行:
#   .venv/bin/mjpython bittle_gemini_direct.py \
#     --prompt "前に3歩進んでから、お座りして、右前足を上げて3回バイバイして" \
#     --viewer
# ---------------------------------------------------------------------------

direct_sandbox = DirectBittleSandbox(record_video=True)
run_pure_direct_agent(
    "後ろ足のだけ2足立ちして、両前足を上げて3回バイバイして",
    direct_sandbox,
)

video_path_part2 = ROOT_DIR / "notebook_part2_direct_biped.mp4"
direct_sandbox.save_video(video_path_part2)
Video(str(video_path_part2), embed=True, width=560)

[Turn 1] Calling low-level tool: set_joint_angles({'transition_s': 0.6, 'lr_knee_deg': 60, 'rr_shoulder_deg': 55, 'rf_knee_deg': 40, 'rf_shoulder_deg': 60, 'lf_knee_deg': 40, 'lf_shoulder_deg': 60, 'rr_knee_deg': 60, 'lr_shoulder_deg': 55, 'hold_s': 0.5})
   -> Result: displacement=None, after_pos={'x': -0.0849, 'y': 0.0001, 'z_above_floor': 0.0568}
[Turn 2] Calling low-level tool: set_joint_angles({'lr_knee_deg': 70, 'transition_s': 0.5, 'rf_knee_deg': 40, 'lf_shoulder_deg': -60, 'hold_s': 0.4, 'rr_shoulder_deg': 70, 'rf_shoulder_deg': -60, 'lf_knee_deg': 40, 'rr_knee_deg': 70, 'lr_shoulder_deg': 70})
   -> Result: displacement=None, after_pos={'x': -0.0836, 'y': -0.0005, 'z_above_floor': 0.0608}
[Turn 3] Calling low-level tool: set_joint_angles({'lr_knee_deg': -20, 'hold_s': 0.5, 'lr_shoulder_deg': 85, 'rr_knee_deg': -20, 'lf_shoulder_deg': 75, 'rf_shoulder_deg': 75, 'rr_shoulder_deg': 85, 'rf_knee_deg': 55, 'transition_s': 0.6, 'lf_knee_deg': 55})
   -> Result: displacement=None, af

---
## Part 3. 自律スキル学習・蓄積・再利用システム (`bittle_gemini_skills.py`)

Part 2 の方法では、未知の動きでもゼロから作れる反面、毎回関節角度や CPG パラメータを計算し直すため API トークンと推論時間がかかります。
そこで Part 3 では、**一度うまく動いた関節角度・キーフレーム列・CPGパラメータを `learned_skills.json` に名前付きスキルとして自律的に保存し、次回から 1 ショットで呼び出す仕組み (`SkillLibrary`)** を導入します。

### スキル学習・蓄積のサイクル
1. **3種類のスキル形式 (`pose`, `keyframes`, `cpg`) で JSON に永続化**
   - `save_pose_skill("sit", ...)`：静止ポーズを保存
   - `save_keyframe_skill("wave_right_paw", ...)`：1回分の基本ジェスチャー波形を保存（実行時に `repeat=N` で N 回反復可能）
   - `save_cpg_skill("trot_forward", ...)`：CPG 歩行パラメータを保存（実行時に `repeat=N` で N 歩移動可能）
2. **プロンプトへの自動注入 (`summary_for_prompt`)**
   - 起動時に `learned_skills.json` のスキル一覧をシステムに自動で読み込ませます。
3. **1ショット実行 (`execute_saved_skill`) と自己改良**
   - すでに習得済みのスキルだけで構成できる指示なら、**最初の 1 ターン目で必要なスキルをすべて連続呼び出し** して即座に完了します。
   - もし未知の動作が含まれていれば、その部分だけ低レベルツールで合成・検証して新たに JSON へ追加します。


In [11]:
# [Part 3 - Step 1] スキルライブラリ (`SkillLibrary`) とスキル管理ツールの読み込み
from bittle_gemini_skills import (
    SkillLibrary,
    SYSTEM_INSTRUCTION as SKILLS_SYSTEM_INSTRUCTION,
    build_direct_tools as build_skill_learning_tools,
    run_direct_gemini_agent as run_skill_learning_agent,
)

# 既存の learned_skills.json の中身を確認
skills_file = ROOT_DIR / "learned_skills.json"
skill_lib = SkillLibrary(skills_file)
print(f"現在保存されているスキル数: {len(skill_lib.skills)}")
print(skill_lib.summary_for_prompt())


現在保存されているスキル数: 0
None yet (Skill library is currently empty).


In [12]:
# [Part 3 - Step 2] 1回目（学習フェーズ）：スキルを空にリセットした状態から動作を生成し、JSONへ自動保存させる
# ※ すでに learned_skills.json がある場合は、別ファイル (notebook_learned_skills.json) でゼロからの学習を確認します。

notebook_skills_path = ROOT_DIR / "notebook_learned_skills.json"
if notebook_skills_path.exists():
    notebook_skills_path.unlink()

nb_skill_lib = SkillLibrary(notebook_skills_path)
learn_sandbox = DirectBittleSandbox(record_video=True)

run_skill_learning_agent(
    prompt="後ろ足のだけ2足立ちして、両前足を上げて3回バイバイして",
    sandbox=learn_sandbox,
    skill_lib=nb_skill_lib,
)

print(f"現在保存されているスキル数: {len(nb_skill_lib.skills)}")
print(nb_skill_lib.summary_for_prompt())

print("\n=== 自動生成・保存された JSON ファイルの中身 ===")
print(notebook_skills_path.read_text(encoding="utf-8"))


[Zero-Preset + Skill-Library Agent] Model: gemini-4-argon
[Skills Loaded] 0 skill(s): []
[User Prompt]   後ろ足のだけ2足立ちして、両前足を上げて3回バイバイして
[Initial State] {'torso_pos_m': {'x': 0.0006, 'y': -0.0, 'z_above_floor': 0.0948}, 'orientation_deg': {'yaw': -0.0, 'pitch': -0.0, 'roll': -0.0}, 'fallen': False, 'commanded_joint_angles_deg': {'rf_shoulder_deg': 0.0, 'rf_knee_deg': 0.0, 'lf_shoulder_deg': 0.0, 'lf_knee_deg': 0.0, 'rr_shoulder_deg': 0.0, 'rr_knee_deg': 0.0, 'lr_shoulder_deg': 0.0, 'lr_knee_deg': 0.0}, 'foot_site_heights_above_floor_mm': {'RF': 24.0, 'LF': 24.1, 'RR': 24.2, 'LR': 24.2}, 'sim_time_s': 0.3}

  -> Executing tool: set_joint_angles({'lf_knee_deg': 40, 'rf_knee_deg': 40, 'transition_s': 0.6, 'rr_shoulder_deg': 65, 'rr_knee_deg': 55, 'lf_shoulder_deg': 70, 'lr_shoulder_deg': 65, 'lr_knee_deg': 55, 'hold_s': 0.5, 'rf_shoulder_deg': 70})
     Result: {'before_pos_m': {'x': 0.0006, 'y': -0.0, 'z_above_floor': 0.0948}, 'after_state': {'torso_pos_m': {'x': -0.0912, 'y': -0.0, 'z_abo

In [13]:
# [Part 3 - Step 3] 2回目（再利用フェーズ）：蓄積されたスキルを使って、順番・回数を変えた指示を一発実行する
# ---------------------------------------------------------------------------
# 【.py スクリプトとしてターミナルから実行する場合のコマンド】
#   # 1. スキルをリセットしてゼロから学習＆保存させる場合:
#   .venv/bin/python bittle_gemini_skills.py --reset-skills \
#     --prompt "前に3歩進んでから、お座りして、右前足を上げて3回バイバイして、最後に伏せて" \
#     --video bittle_skill_learn_run1.mp4
#
#   # 2. 保存された learned_skills.json を使って別の指示を高速実行する場合:
#   .venv/bin/python bittle_gemini_skills.py \
#     --prompt "お座りして、右前足で4回バイバイしてから、前に5歩進んで、最後に伏せて" \
#     --video bittle_skill_reuse_run2.mp4
#
#   # 3. 3D Viewer で見ながら新しいスキル（例：腕立て伏せ）を追加学習させる場合:
#   .venv/bin/mjpython bittle_gemini_skills.py \
#     --prompt "腕立て伏せを3回してからお座りして" \
#     --viewer
# ---------------------------------------------------------------------------

reuse_sandbox = DirectBittleSandbox(record_video=True)

run_skill_learning_agent(
    prompt="後ろ足のだけ2足立ちして、両前足を上げて3回バイバイして",
    sandbox=reuse_sandbox,
    skill_lib=nb_skill_lib,
)

print(f"現在保存されているスキル数: {len(nb_skill_lib.skills)}")
print(nb_skill_lib.summary_for_prompt())
video_path_part3 = ROOT_DIR / "notebook_part3_skill_reuse_biped.mp4"
reuse_sandbox.save_video(video_path_part3)
Video(str(video_path_part3), embed=True, width=560)


[Zero-Preset + Skill-Library Agent] Model: gemini-4-argon
[Skills Loaded] 2 skill(s): ['stand_on_hind_legs', 'wave_both_paws_on_hind_legs']
[User Prompt]   後ろ足のだけ2足立ちして、両前足を上げて3回バイバイして
[Initial State] {'torso_pos_m': {'x': 0.0006, 'y': -0.0, 'z_above_floor': 0.0948}, 'orientation_deg': {'yaw': -0.0, 'pitch': -0.0, 'roll': -0.0}, 'fallen': False, 'commanded_joint_angles_deg': {'rf_shoulder_deg': 0.0, 'rf_knee_deg': 0.0, 'lf_shoulder_deg': 0.0, 'lf_knee_deg': 0.0, 'rr_shoulder_deg': 0.0, 'rr_knee_deg': 0.0, 'lr_shoulder_deg': 0.0, 'lr_knee_deg': 0.0}, 'foot_site_heights_above_floor_mm': {'RF': 24.0, 'LF': 24.1, 'RR': 24.2, 'LR': 24.2}, 'sim_time_s': 0.3}

  -> Executing tool: execute_saved_skill({'repeat': 1, 'skill_name': 'stand_on_hind_legs'})
     Result: {'num_keyframes_executed': 2, 'displacement_m': {'dx': -0.0548, 'dy': 0.0002}, 'before_pos_m': {'x': 0.0006, 'y': -0.0, 'z_above_floor': 0.0948}, 'after_state': {'torso_pos_m': {'x': -0.0542, 'y': 0.0002, 'z_above_floor': 0.0467}, '

In [14]:
reuse_sandbox = DirectBittleSandbox(record_video=True)

run_skill_learning_agent(
    prompt="後ろ足のだけ2足立ちして、両前足を上げて3回バイバイした後、2本の後ろ足のだけで2足歩行して",
    sandbox=reuse_sandbox,
    skill_lib=nb_skill_lib,
)

print(f"現在保存されているスキル数: {len(nb_skill_lib.skills)}")
print(nb_skill_lib.summary_for_prompt())

video_path_part3 = ROOT_DIR / "notebook_part3_skill_reuse_biped_walk.mp4"
reuse_sandbox.save_video(video_path_part3)
Video(str(video_path_part3), embed=True, width=560)


[Zero-Preset + Skill-Library Agent] Model: gemini-4-argon
[Skills Loaded] 2 skill(s): ['stand_on_hind_legs', 'wave_both_paws_on_hind_legs']
[User Prompt]   後ろ足のだけ2足立ちして、両前足を上げて3回バイバイした後、2本の後ろ足のだけで2足歩行して
[Initial State] {'torso_pos_m': {'x': 0.0006, 'y': -0.0, 'z_above_floor': 0.0948}, 'orientation_deg': {'yaw': -0.0, 'pitch': -0.0, 'roll': -0.0}, 'fallen': False, 'commanded_joint_angles_deg': {'rf_shoulder_deg': 0.0, 'rf_knee_deg': 0.0, 'lf_shoulder_deg': 0.0, 'lf_knee_deg': 0.0, 'rr_shoulder_deg': 0.0, 'rr_knee_deg': 0.0, 'lr_shoulder_deg': 0.0, 'lr_knee_deg': 0.0}, 'foot_site_heights_above_floor_mm': {'RF': 24.0, 'LF': 24.1, 'RR': 24.2, 'LR': 24.2}, 'sim_time_s': 0.3}

  -> Executing tool: list_saved_skills({})
     Result: {'skills': {'stand_on_hind_legs': {'name': 'stand_on_hind_legs', 'kind': 'keyframes', 'description': 'Stand/balance on hind legs only with both front paws lifted high in the air (2-step transition: first pitch nose up using front legs while positioning rear legs 